# M5A1 - Inspeção Visual de Itens em Esteira de Manufatura

Na prática de hoje vamos refinar um modelo para a tarefa de inspeção visual.

Esse notebook está estruturado da seguinte forma.

- Introdução
- Carregar Base de Dados
- Refinar Modelo
- Próximos passos
- Atividade Complementares

## Introdução

Instalação para os que ainda não possuem a biblioteca instalada.

In [3]:
!pip install torch torchvision huggingface_hub ultralytics

Importar as bibliotecas

In [4]:
import yaml
import os

from ultralytics import YOLO
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil
from IPython.display import Video

/home/joaoferreira/trilha_visao_computacional/.env-trilha-vc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carregar Base de Dados

A primeira tarefa para refinar um modelo é criar a base de dados.

Referência: https://huggingface.co/datasets/johnatanvq/fruits-dataset

In [5]:
# 1) Baixar somente o subdiretório fruitsData/
local_repo_dir = snapshot_download(
    repo_id="johnatanvq/fruits-dataset",
    repo_type="dataset",
    allow_patterns=["fruitsData/**"],  # baixa só essa pasta
)

print("Arquivos baixados em:", local_repo_dir)

# 2) Mover/copiar para uma pasta final estilo ImageFolder (se quiser customizar o caminho)
src = Path(local_repo_dir) / "fruitsData"
dst = Path("data/fruits")  # pasta final onde você quer o ImageFolder

# Copiando arquivos para dst.
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print("ImageFolder pronto em:", dst)


Fetching 322 files: 100%|██████████| 322/322 [00:00<00:00, 13243.96it/s]


Arquivos baixados em: /home/joaoferreira/.cache/huggingface/hub/datasets--johnatanvq--fruits-dataset/snapshots/2e9cf7d297327f8c1890ac1616be62c87d6fe5f5
ImageFolder pronto em: data/fruits


In [6]:
def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
  # Lê o arquivos "classes.txt".
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Cria o dicionário a ser salvo.
  data = {
      'path': 'data/fruits',
      'train': 'images',
      'val': 'images',
      'nc': number_of_classes,
      'names': classes
  }

  # Escreve o arquivo YAML.
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Chama a função.
create_data_yaml("data/fruits/classes.txt", "yolo_train.yaml")

Created config file at yolo_train.yaml


In [8]:
# Carrega o modelo pré-treinado.
model = YOLO("yolo11n.pt")

# Treina o modelo utilizando as informações do arquivo YAML.
# Definimos também a quantidade de épocas, o batch, e o tamanho das imagens.
results = model.train(data="./yolo_train.yaml", project="praticas/modulo_5/aula_1", epochs=10, batch=2, imgsz=480)

New https://pypi.org/project/ultralytics/8.3.243 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.240 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 5804MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_train.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=Fals

In [ ]:
Video("../assets/modulo5/video.mov")

In [ ]:
model.predict("../assets/modulo5/video.mov", save=True, project="praticas/modulo_5/aula_1")

In [ ]:
Video("/home/joaoferreira/trilha_visao_computacional/praticas/modulo_5/aula_1/praticas/modulo_5/aula_1/predict/video.avi")

: 

## Próximos Passos e Referências

Nas próximas práticas vamos continuar trabalhando com problemas reais que envolvem Visão Computacional.

Uma lista não exaustiva de referências segue:

- https://docs.ultralytics.com/modes/train/
- https://docs.ultralytics.com/modes/predict/
- https://huggingface.co/datasets/johnatanvq/fruits-dataset
- https://huggingface.co/datasets
- https://pytorch.org/
- https://docs.pytorch.org/vision/main/models.html
- https://opencv.org/
- https://learnopencv.com/blogs/
- https://pyimagesearch.com/

## Atividades Complementares (Opcional)

- [ ] Tente alterar a base de dados e veja se o modelo continua funcionando?
- [ ] Tente alterar alguns hiperparâmetros de treinamento, batch e resolução da imagens e veja como isso altera os resultados.

---

### Atividade 1 — Outra base de dados

Testei o mesmo pipeline com o dataset `coco8` que vem embutido no Ultralytics e baixa automaticamente. Tem imagens do COCO com 80 classes, bem diferente das frutas originais. A ideia era ver se o modelo ainda treinava normalmente com classes completamente diferentes.

### Atividade 2 — Alterando hiperparâmetros

Fiz dois experimentos com batch e resolução diferentes. Batch menor com resolução maior tende a capturar mais detalhes mas é mais lento. Batch maior com resolução menor é mais rápido mas perde um pouco nas detecções de objetos pequenos.

In [ ]:
# Testando com outro dataset — coco8 vem embutido no Ultralytics e baixa sozinho
# Tem imagens do COCO com 80 classes, bem diferente das frutas
model_nova_base = YOLO("yolo11n.pt")
results_nova_base = model_nova_base.train(
    data="coco8.yaml",
    project="praticas/modulo_5/aula_1_nova_base",
    epochs=10,
    batch=2,
    imgsz=480
)
print("Treinou sem problema com base diferente. O pipeline é o mesmo — só o data= muda.")

In [ ]:
# Experimento 1: batch menor, resolução maior — captura mais detalhes, mas é mais lento
print("=== Experimento 1: batch=1, imgsz=640 ===")
model_exp1 = YOLO("yolo11n.pt")
results_exp1 = model_exp1.train(
    data="./yolo_train.yaml",
    project="praticas/modulo_5/aula_1_hp",
    name="batch1_640",
    epochs=5,
    batch=1,
    imgsz=640
)

# Experimento 2: batch maior, resolução menor — mais rápido, perde detalhes finos
print("\n=== Experimento 2: batch=4, imgsz=320 ===")
model_exp2 = YOLO("yolo11n.pt")
results_exp2 = model_exp2.train(
    data="./yolo_train.yaml",
    project="praticas/modulo_5/aula_1_hp",
    name="batch4_320",
    epochs=5,
    batch=4,
    imgsz=320
)